# Drive -> SRT (ruso) con `faster-whisper-xxl` (large-v2)

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con el binario standalone **Faster-Whisper-XXL r245.4** de Purfview (large-v2 + extracción de voz Kim2), y guarda los `.srt` en tu Drive (carpeta **Subs_RU**).

**Antes de correr:** `Runtime -> Change runtime type -> T4 GPU`.

## Dos modos (los dos repartren el lote sin que las cuentas tengan que coordinarse)

- **Celda 1 — automático (reparto fijo / stride).** Editás 2 números (cuántas cuentas hay y cuál es **esta**) y corrés esa única celda en cada cuenta. Cada una procesa un subconjunto **intercalado** (cuenta 1: videos 1, 9, 17…; cuenta 2: 2, 10, 18…). No hay coordinación entre cuentas, así que es **imposible que dos toquen el mismo video**. Es lo recomendado para los 500+.
- **Celdas 2 + 3 — manual por rango en bloque.** Si preferís bloques contiguos (`1-100`, `101-200`…): corré el setup (celda 2), elegí el rango y transcribí (celda 3).

> **Por qué no una "cola compartida" con locks en Drive:** no funciona. El Drive montado en Colab no le muestra a una cuenta los archivos que otra crea en una carpeta compartida (caché que no se invalida), y peor, Drive permite dos archivos con el mismo nombre (los marca con ` (1)`), así que un lock-por-archivo no sirve para avisar "esto ya está tomado". El reparto fijo evita el problema de raíz: ninguna cuenta necesita ver lo que hacen las otras.

Los tres usan los mismos parámetros de Whisper y guardan el SRT con el mismo nombre del video (`pepe.mp4` -> `pepe.srt`).

## Parámetros

```
-m large-v2                 -l ru                 --task transcribe
--initial_prompt None       --reprompt False
--condition_on_previous_text False
--hallucination_silence_threshold 4
--compute_type float16      --temperature 0       --beam_size 5
--vad_filter True
--ff_vocal_extract mdx_kim2 --voc_device cuda     --ff_loudnorm
-f srt                      --max_line_width 200  --max_line_count 1   --sentence
```

> **Limpieza de audio:** `--ff_vocal_extract mdx_kim2` separa la voz (MDX-Net Kim2) en la misma GPU antes de transcribir; la primera corrida del runtime baja el modelo Kim2 (cientos de MB) y sube el tiempo por video (~1.5x-2x).
> **Diarización opcional:** `--diarize pyannote_v3.1 --diarize_device cuda` están comentados en `build_cmd`; descomentalos si querés etiquetas de hablante.
> **Salida:** tu comando local usa `-o source` (SRT al lado del video). Acá va a `MyDrive/Subs_RU/<stem>.srt` vía carpeta temporal.


## 1) Modo automático — reparto fijo entre cuentas (RECOMENDADO)

Corré **solo esta celda** en cada una de tus cuentas de Colab. Lo único que cambiás en cada cuenta son los dos números del principio:

```
N_MAQUINAS = 8     # cuántas cuentas están corriendo en total
POSICION   = 1     # cuál es ESTA cuenta: 1, 2, 3, ... hasta N_MAQUINAS
```

Cada cuenta procesa un subconjunto **intercalado** de la lista (ordenada natural): la cuenta `POSICION` hace los videos `POSICION, POSICION+N, POSICION+2N, …`. Con 8 cuentas: la 1 hace 1, 9, 17, 25…; la 2 hace 2, 10, 18…; etc.

- **Imposible que dos cuentas toquen el mismo video** — no hay coordinación ni locks, cada una sabe exactamente qué le toca.
- El intercalado reparte parejo los videos largos y cortos (no se amontonan en una cuenta).
- **Re-run seguro:** si una cuenta se corta, la relanzás con el **mismo** `POSICION` y retoma su subconjunto saltando los que ya tienen SRT.
- Si una cuenta queda libre y querés que ayude con lo que falta, cambiale el `POSICION`/`N_MAQUINAS` para que apunte a otro tramo, o pasala al modo manual (celda 2-3) con un rango puntual.


In [ ]:
!apt-get -qq install -y ffmpeg p7zip-full > /dev/null

import os, time, re, subprocess, shutil
from pathlib import Path
from google.colab import drive
import torch

# --- Descargar + extraer faster-whisper-xxl (una sola vez por sesion) ---
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"

if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, \
            f"Descarga fallo o quedo incompleta (size={ARCHIVE.stat().st_size if ARCHIVE.exists() else 0})"
    print("Extrayendo (puede tardar 1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), f"Extraccion fallo - no aparecio {EXE}"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)

# --- Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# --- Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: 'float16' y la extraccion de voz en 'cuda' van a fallar. Activa T4 GPU.")

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    # Orden "humano": los tramos de digitos se comparan como enteros,
    # asi 2 < 10 < 100 (no lexicografico, donde "100" caeria antes que "2").
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS),
                key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
pending = total - already
print(f"\n{total} archivo(s) en '{INPUT_DIR.name}'  |  {already} con SRT  |  {pending} pendientes")
print("Primeros 5:")
for i, p in enumerate(inputs[:5], 1):
    mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
    print(f"  {i:>4}. [{mark}] {p.name}")
if total > 10:
    print("  ...")
    for i, p in enumerate(inputs[-3:], total - 2):
        mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
        print(f"  {i:>4}. [{mark}] {p.name}")

# --- Carpeta temporal + armador del comando (flags en UN solo lugar) ---
TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        # --- Modelo / idioma ---
        "--model", "large-v2",
        "--language", "ru",
        "--task", "transcribe",
        # --- Decoder ---
        "--initial_prompt", "None",
        "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16",
        "--temperature", "0",
        "--beam_size", "5",
        "--vad_filter", "True",
        # --- Audio: limpieza para musica / ruido / multi-hablante ---
        "--ff_vocal_extract", "mdx_kim2",
        "--voc_device", "cuda",
        "--ff_loudnorm",
        # --- Diarizacion: descomenta SOLO si necesitas etiquetas de hablante (pasada extra pesada) ---
        # "--diarize", "pyannote_v3.1",
        # "--diarize_device", "cuda",
        # --- Salida ---
        "--max_line_width", "200",
        "--max_line_count", "1",
        "--sentence",
        "--output_dir", str(TMP_OUT),
        "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    """Corre el CLI para un video y mueve el SRT a Drive. Devuelve 'done'|'failed'."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"

# ===== Modo automatico: reparto fijo entre cuentas (stride) =====
import numpy as np
from IPython.display import Audio, display

# >>> EDITA ESTOS DOS NUMEROS EN CADA CUENTA <<<
N_MAQUINAS = 8     # cuantas cuentas/Colabs estan corriendo en total
POSICION   = 1     # cual es ESTA cuenta: 1, 2, 3, ... hasta N_MAQUINAS

assert 1 <= POSICION <= N_MAQUINAS, f"POSICION debe estar entre 1 y {N_MAQUINAS}"

# Esta cuenta procesa los videos POSICION, POSICION+N, POSICION+2N, ... (1-indexado)
mi_lote = inputs[POSICION - 1::N_MAQUINAS]
print(f"Cuenta {POSICION}/{N_MAQUINAS}: me tocan {len(mi_lote)} de {total} videos "
      f"(posiciones {POSICION}, {POSICION+N_MAQUINAS}, {POSICION+2*N_MAQUINAS}, ...).")

done = skipped = failed = 0
t_global = time.time()
for vid in mi_lote:
    idx = inputs.index(vid) + 1
    if (OUTPUT_DIR / f"{vid.stem}.srt").exists():
        print(f"\n[{idx}/{total}] SALTADO (ya existe): {vid.stem}.srt")
        skipped += 1
        continue
    print(f"\n[{idx}/{total}] Procesando: {vid.name}")
    try:
        outcome = transcribe_one(vid, f"{idx}/{total}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"
    done    += outcome == "done"
    failed  += outcome == "failed"

print(f"\n=== Fin cuenta {POSICION}/{N_MAQUINAS} ===")
print(f"  transcritos: {done}")
print(f"  saltados:    {skipped}")
print(f"  fallidos:    {failed}")
print(f"  tiempo: {(time.time()-t_global)/60:.1f} min")
print(f"\nSRT en: {OUTPUT_DIR}")

# Ruidito final
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))


## 2) Setup (modo manual por rango)

Para el modo manual. Corré esta celda: instala el binario, monta Drive, lista `Host Videos`, te muestra el total + un reparto sugerido, y te da un cuadro para escribir el rango (ej. `1-100`). Después pasá a la celda 3.

> Si vas a usar el **modo automático (celda 1)**, ignorá las celdas 2 y 3.


In [ ]:
!apt-get -qq install -y ffmpeg p7zip-full > /dev/null

import os, time, re, subprocess, shutil
from pathlib import Path
from google.colab import drive
import torch

# --- Descargar + extraer faster-whisper-xxl (una sola vez por sesion) ---
XXL_URL = "https://github.com/Purfview/whisper-standalone-win/releases/download/Faster-Whisper-XXL/Faster-Whisper-XXL_r245.4_linux.7z"
INSTALL_DIR = Path("/content/whisper")
EXE = INSTALL_DIR / "Faster-Whisper-XXL" / "faster-whisper-xxl"
ARCHIVE = INSTALL_DIR / "fwxxl.7z"

if EXE.exists():
    print(f"Binario ya instalado: {EXE}")
else:
    INSTALL_DIR.mkdir(parents=True, exist_ok=True)
    if not ARCHIVE.exists():
        print("Descargando Faster-Whisper-XXL r245.4 para Linux (~1.54 GB, una sola vez)...")
        rc = subprocess.call(["wget", "-q", "--show-progress", "-O", str(ARCHIVE), XXL_URL])
        assert rc == 0 and ARCHIVE.stat().st_size > 100 * 1024 * 1024, \
            f"Descarga fallo o quedo incompleta (size={ARCHIVE.stat().st_size if ARCHIVE.exists() else 0})"
    print("Extrayendo (puede tardar 1-2 min)...")
    rc = subprocess.call(["7z", "x", "-y", "-bso0", "-bsp0", f"-o{INSTALL_DIR}", str(ARCHIVE)])
    assert rc == 0 and EXE.exists(), f"Extraccion fallo - no aparecio {EXE}"
    ARCHIVE.unlink(missing_ok=True)
os.chmod(EXE, 0o755)

# --- Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# --- Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")
assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    print("[WARN] sin GPU: 'float16' y la extraccion de voz en 'cuda' van a fallar. Activa T4 GPU.")

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}

def natural_key(p):
    # Orden "humano": los tramos de digitos se comparan como enteros,
    # asi 2 < 10 < 100 (no lexicografico, donde "100" caeria antes que "2").
    return [int(t) if t.isdigit() else t.lower()
            for t in re.split(r"(\d+)", p.name)]

inputs = sorted((p for p in INPUT_DIR.iterdir()
                 if p.is_file() and p.suffix.lower() in VIDEO_EXTS),
                key=natural_key)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

total = len(inputs)
already = sum(1 for p in inputs if (OUTPUT_DIR / f"{p.stem}.srt").exists())
pending = total - already
print(f"\n{total} archivo(s) en '{INPUT_DIR.name}'  |  {already} con SRT  |  {pending} pendientes")
print("Primeros 5:")
for i, p in enumerate(inputs[:5], 1):
    mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
    print(f"  {i:>4}. [{mark}] {p.name}")
if total > 10:
    print("  ...")
    for i, p in enumerate(inputs[-3:], total - 2):
        mark = "ok" if (OUTPUT_DIR / f"{p.stem}.srt").exists() else "--"
        print(f"  {i:>4}. [{mark}] {p.name}")

# --- Carpeta temporal + armador del comando (flags en UN solo lugar) ---
TMP_OUT = Path("/content/srt_tmp"); TMP_OUT.mkdir(exist_ok=True)

def build_cmd(vid):
    return [
        str(EXE), str(vid),
        # --- Modelo / idioma ---
        "--model", "large-v2",
        "--language", "ru",
        "--task", "transcribe",
        # --- Decoder ---
        "--initial_prompt", "None",
        "--reprompt", "False",
        "--condition_on_previous_text", "False",
        "--hallucination_silence_threshold", "4",
        "--compute_type", "float16",
        "--temperature", "0",
        "--beam_size", "5",
        "--vad_filter", "True",
        # --- Audio: limpieza para musica / ruido / multi-hablante ---
        "--ff_vocal_extract", "mdx_kim2",
        "--voc_device", "cuda",
        "--ff_loudnorm",
        # --- Diarizacion: descomenta SOLO si necesitas etiquetas de hablante (pasada extra pesada) ---
        # "--diarize", "pyannote_v3.1",
        # "--diarize_device", "cuda",
        # --- Salida ---
        "--max_line_width", "200",
        "--max_line_count", "1",
        "--sentence",
        "--output_dir", str(TMP_OUT),
        "--output_format", "srt",
    ]

def transcribe_one(vid, label):
    """Corre el CLI para un video y mueve el SRT a Drive. Devuelve 'done'|'failed'."""
    final_srt = OUTPUT_DIR / f"{vid.stem}.srt"
    for f in TMP_OUT.glob(f"{vid.stem}.*"):
        try: f.unlink()
        except OSError: pass
    t0 = time.time()
    result = subprocess.run(build_cmd(vid), capture_output=True, text=True)
    cli_srt = TMP_OUT / f"{vid.stem}.srt"
    if result.returncode == 0 and cli_srt.exists():
        shutil.move(str(cli_srt), str(final_srt))
        n = sum(1 for L in final_srt.read_text(encoding="utf-8").splitlines() if L.strip().isdigit())
        print(f"   ok {label}: {n} cues, {time.time()-t0:.1f}s -> {final_srt.name}")
        return "done"
    print(f"   x {label}: CLI exit {result.returncode}; sin SRT.")
    err = (result.stderr or "")[-400:]
    if err: print(f"   stderr:\n{err}")
    return "failed"

import ipywidgets as widgets
from IPython.display import display

# --- Reparto sugerido entre 5 cuentas ---
N_ACCOUNTS = 5
chunk = (total + N_ACCOUNTS - 1) // N_ACCOUNTS
print(f"\nReparto sugerido para {N_ACCOUNTS} cuentas (~{chunk} videos cada una):")
for k in range(N_ACCOUNTS):
    a = k * chunk + 1
    b = min((k + 1) * chunk, total)
    if a > total: break
    print(f"   cuenta {k+1} -> {a}-{b}")

# --- Cuadro de input para el rango ---
range_box = widgets.Text(
    value=f"1-{min(100, total)}",
    placeholder="ej: 1-100  (o 'all' para todo)",
    description="Rango:",
    layout=widgets.Layout(width="60%"),
    style={"description_width": "60px"},
)
print(f"\nElegi el rango (1..{total}) y pasa a la celda 3:")
display(range_box)


## 3) Transcribir un rango (modo manual)

Lee el rango del cuadro de la celda 2 y transcribe solo esos videos. Salta los que ya tienen SRT en Drive.


In [ ]:
import numpy as np
from IPython.display import Audio, display

# --- Parsear el rango del cuadro de la celda 2 ---
assert "range_box" in globals(), "Primero corre la celda 2 (setup manual)."
spec = (range_box.value or "").strip().lower()
if spec in ("", "all"):
    start, end = 1, total
else:
    m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", spec) or re.fullmatch(r"(\d+)", spec)
    assert m, f"Rango invalido: {spec!r}. Usa '1-100' o '50' o 'all'."
    if m.lastindex == 1:
        start = end = int(m.group(1))
    else:
        start, end = int(m.group(1)), int(m.group(2))
start = max(1, start); end = min(total, end)
assert start <= end, f"Rango vacio despues de recortar a 1..{total}: {start}-{end}"

batch = inputs[start-1:end]
print(f"Procesando {len(batch)} archivo(s): #{start} a #{end} de {total}.")

done = skipped = failed = 0
t_global = time.time()
for offset, vid in enumerate(batch):
    i = start + offset
    if (OUTPUT_DIR / f"{vid.stem}.srt").exists():
        print(f"\n[{i}/{end}] SALTADO (ya existe en Drive): {vid.stem}.srt")
        skipped += 1
        continue
    print(f"\n[{i}/{end}] Procesando: {vid.name}")
    try:
        outcome = transcribe_one(vid, f"{i}/{end}")
    except Exception as ex:
        print(f"   x excepcion: {ex}"); outcome = "failed"
    done    += outcome == "done"
    failed  += outcome == "failed"

print(f"\n=== Resumen del rango {start}-{end} ===")
print(f"  procesados: {done}")
print(f"  saltados:   {skipped}")
print(f"  fallidos:   {failed}")
print(f"  tiempo total: {(time.time()-t_global)/60:.1f} min")
print(f"\nSRT en: {OUTPUT_DIR}")

# Ruidito final
sr = 22050; out_audio = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    out_audio = np.concatenate([out_audio, (0.3*np.exp(-3*t)*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out_audio, rate=sr, autoplay=True))
